# Analyse: Gesichtsausrichtung


In [ ]:
import sys
!{sys.executable} -m pip uninstall -y mediapipe mediapipe-silicon
!{sys.executable} -m pip install "mediapipe==0.10.14"

In [ ]:
import mediapipe as mp
print(mp.__version__)
print(mp.solutions)


In [ ]:
import sys
!{sys.executable} -m pip install numpy pandas opencv-python tqdm


In [2]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from tqdm import tqdm


In [3]:
DATA_DIR =  Path('../../data')

COMBINED_CSV = DATA_DIR / '03_datasets' / 'influencer_balanced' / '01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '04_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_angle_face_orientation.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'
SOURCE_FILTER = None          # e.g. ['ai', 'real'] or None
DEFAULT_MAX_FRAMES_PER_VIDEO = 60 

# Mindestscore für akzeptierte Gesichtserkennung

MIN_DET_CONF = 0.8
# Modell für größere Distanzbereiche
MODEL_SELECTION = 1           # 0: short-range, 1: full-range


df = pd.read_csv(COMBINED_CSV)


In [4]:
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'


def get_video_id(row) -> str:
    value = row.get(video_id_col, None)
    if pd.isna(value):
        return ''
    return str(value)


def get_duration_seconds(row):
    # Mögliche Spaltennamen für die Videodauer
    for col in ('video_duration_seconds', 'duration_seconds', 'duration', 'video_duration'):
        if col in row.index:
            value = row.get(col, np.nan)
            if pd.notna(value):
                return value
    return np.nan


def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()


# Frame-Ordner erneut prüfen
video_ids = df[video_id_col].astype(str)
df['has_frames'] = [has_frames(video_id) for video_id in video_ids]

missing_ids = video_ids[~df['has_frames']].tolist()
print(f'Total videos in CSV: {len(df)}')
print(f'Videos with frame folder: {df["has_frames"].sum()}')
print(f'Videos missing frame folder: {len(missing_ids)}')

if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

# Nur Videos mit vorhandenen Frames weiterverarbeiten
df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')



Total videos in CSV: 500
Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


In [5]:
# MediaPipe-Gesichtserkennung vorbereiten
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

def estimate_pitch_yaw_from_detection(detection, w: int, h: int):
    """
    Approximate pitch/yaw from FaceDetection keypoints.
    Returns (pitch_deg, yaw_deg) or None.
    """
    try:
        nose = mp_face_detection.get_key_point(detection, mp_face_detection.FaceKeyPoint.NOSE_TIP)
        left_eye = mp_face_detection.get_key_point(detection, mp_face_detection.FaceKeyPoint.LEFT_EYE)
        right_eye = mp_face_detection.get_key_point(detection, mp_face_detection.FaceKeyPoint.RIGHT_EYE)
        mouth = mp_face_detection.get_key_point(detection, mp_face_detection.FaceKeyPoint.MOUTH_CENTER)
    except Exception:
        return None

    nose = np.array([nose.x * w, nose.y * h], dtype=float)
    left_eye = np.array([left_eye.x * w, left_eye.y * h], dtype=float)
    right_eye = np.array([right_eye.x * w, right_eye.y * h], dtype=float)
    mouth = np.array([mouth.x * w, mouth.y * h], dtype=float)

    eye_mid = (left_eye + right_eye) / 2.0
    eye_dist = np.linalg.norm(left_eye - right_eye)
    eye_to_mouth = np.linalg.norm(mouth - eye_mid)

    if eye_dist < 1e-6 or eye_to_mouth < 1e-6:
        return None

    # Gierwinkel: horizontaler Nasenversatz relativ zur Augenmitte, normiert über den Augenabstand
    yaw_norm = (nose[0] - eye_mid[0]) / (eye_dist / 2.0)
    yaw = float(np.clip(yaw_norm * 30.0, -60.0, 60.0))

    # Nickwinkel: vertikale Nasenposition zwischen Augen- und Mundlinie
    pitch_norm = (nose[1] - eye_mid[1]) / eye_to_mouth
    pitch = float(np.clip((pitch_norm - 0.45) * 80.0, -60.0, 60.0))

    return pitch, yaw

def categorize_orientation(pitch, yaw, pitch_thr=12.0, yaw_thr=12.0):
    if pitch is None or yaw is None:
        return "Unknown"
    if yaw > yaw_thr:
        return "Right"
    if yaw < -yaw_thr:
        return "Left"
    if pitch > pitch_thr:
        return "Down"
    if pitch < -pitch_thr:
        return "Up"
    return "Frontal"


def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []

    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        return []

    # Ziel: ein Frame pro Videosekunde
    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, len(frame_files)))
    step = max(1, len(frame_files) // target_frames)
    return frame_files[::step][:target_frames]



In [6]:
# Alle Videos verarbeiten
video_pitch_means = []
video_yaw_means = []
video_orientation = []
video_faces_detected = []

with mp_face_detection.FaceDetection(
    model_selection=MODEL_SELECTION,
    min_detection_confidence=MIN_DET_CONF
) as face_detection:

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing videos"):
        vid = get_video_id(row)
        duration_seconds = get_duration_seconds(row)
        frame_paths = sample_frame_paths(vid, duration_seconds=duration_seconds)

        pitch_vals = []
        yaw_vals = []
        face_count = 0

        for idx, file in enumerate(frame_paths):
            image = cv2.imread(str(file))
            if image is None:
                continue

            # BGR nach RGB umwandeln und MediaPipe-Gesichtserkennung ausführen
            results = face_detection.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            if not results.detections:
                continue

            # Erkennung mit höchster Konfidenz auswählen
            best_det = max(results.detections, key=lambda d: d.score[0] if d.score else 0.0)
            face_count += 1

            est = estimate_pitch_yaw_from_detection(best_det, image.shape[1], image.shape[0])
            if est is not None:
                pitch, yaw = est
                pitch_vals.append(pitch)
                yaw_vals.append(yaw)


        if pitch_vals:
            p_mean = float(np.mean(pitch_vals))
            y_mean = float(np.mean(yaw_vals))
            label = categorize_orientation(p_mean, y_mean)
        else:
            p_mean = np.nan
            y_mean = np.nan
            label = "Unbestimmt"

        video_pitch_means.append(p_mean)
        video_yaw_means.append(y_mean)
        video_orientation.append(label)
        video_faces_detected.append(face_count)



Processing videos:   0%|          | 0/500 [00:00<?, ?it/s]c:\Users\hanna\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Processing videos: 100%|██████████| 500/500 [02:59<00:00,  2.79it/s]


In [7]:
# Ergebnisse speichern
df["face_pitch_mean"] = video_pitch_means
df["face_yaw_mean"] = video_yaw_means
df["angle_face_orientation"] = video_orientation
df["detected_face_frames"] = video_faces_detected

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved: {OUTPUT_CSV}")
df[["angle_face_orientation", "face_pitch_mean", "face_yaw_mean", "detected_face_frames"]].head(20)


Saved: ..\..\data\04_analysis_results\visual_features\04_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_angle_face_orientation.csv


,angle_face_orientation,face_pitch_mean,face_yaw_mean,detected_face_frames
0,Frontal,0.813416,-6.612124,7
1,Frontal,6.303660,11.687937,10
2,Down,13.049146,-0.242938,5
3,Frontal,2.956885,-1.673240,7
4,Frontal,9.216121,4.903319,5
5,Frontal,7.163271,0.169926,4
6,Left,2.806952,-37.351891,8
7,Left,6.652599,-23.672119,8
8,Frontal,3.959562,9.017248,6
9,Frontal,2.245435,-7.769282,8
